# Publicación de resultados de detección de anomalías en contratos electrónicos de SECOP II

Este notebook prepara los resultados del modelo Isolation Forest, utilizando la configuración C1 definida en los experimentos anteriores (script 09_...). Se generan puntuaciones y clasificaciones para los 82.457 contratos aptos para el análisis, mientras que los 507 registros excluidos se documentan por separado. Los resultados permiten identificar contratos con comportamientos estadísticamente atípicos, sin que esto represente automáticamente una irregularidad en la contratación.


## 1. Configuración

Se define el periodo de análisis, la versión del modelo y las rutas donde se almacenarán los resultados. La escritura en PostgreSQL permanece desactivada mientras se verifican las puntuaciones y clasificaciones generadas.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

NB_DIR = Path("/home/jovyan/work/spark-apps/notebooks")
if not NB_DIR.exists():
    NB_DIR = Path.cwd().resolve()
    if NB_DIR.name == "ml":
        NB_DIR = NB_DIR.parent
SRC = NB_DIR.parent / "src"
sys.path.insert(0, str(SRC))

from transformation.spark_sesion import _asegurar_pyspark, crear_spark

_asegurar_pyspark()
from ml.publicacion import ejecutar
from ml.scoring import N_APTOS, N_EXCLUIDOS, VERSION_MODELO

PERIODO = "2026-09"
ESCRIBIR = False
SALIDAS = NB_DIR / "salidas" / "ml" / PERIODO / "contratos"
SALIDAS.mkdir(parents=True, exist_ok=True)

print("Periodo:", PERIODO)
print("version_modelo:", VERSION_MODELO)
print("ESCRIBIR:", ESCRIBIR)
print("Salidas:", SALIDAS)
print("PGUSER definido:", bool(os.environ.get("PGUSER")))


## 2. Generación de resultados del modelo

Se consultan los contratos almacenados en PostgreSQL y se generan las puntuaciones de anomalía mediante Isolation Forest, utilizando la configuración C1. También se preparan los registros excluidos y la información necesaria para identificar la ejecución realizada

In [ ]:
spark = crear_spark("nb10-publicacion-ml")
print("Spark:", spark.version, spark.sparkContext.master)
res = ejecutar(spark, periodo=PERIODO, escribir=ESCRIBIR, salidas=SALIDAS)
spark.stop()
ej = res["ejecucion"]
display(pd.DataFrame([ej]))
assert ej["n_aptos"] == N_APTOS
assert ej["n_excluidos"] == N_EXCLUIDOS
assert ej["n_top1_if"] == 825
assert ej["n_top5_if"] == 4123
assert res["publicado"] is None
print("id_ejecucion:", ej["id_ejecucion"])


## 3. Validación y comparación de resultados

Se verifica la cantidad de contratos evaluados y excluidos, así como la identificación de los contratos con las puntuaciones más atípicas. También se comparan las clasificaciones de Isolation Forest con las obtenidas mediante el valor económico y el percentil 99 por modalidad de contratación.

In [ ]:
scores = res["scores"]
exc = res["excluidos"]
print("scores", len(scores), "excluidos", len(exc))
print("intersección ids", len(set(scores["id_contrato"]) & set(exc["id_contrato"])))
print("top1_if", int(scores["top1_if"].sum()), "top5_if", int(scores["top5_if"].sum()))
print("top1_logvalor", int(scores["top1_logvalor"].sum()), "p99", int(scores["flag_p99_modalidad"].sum()))
print("ambos top5 IF y logvalor", int((scores["top5_if"] & scores["top5_logvalor"]).sum()))
display(exc["motivo_exclusion"].value_counts().rename("n").reset_index())
display(scores.nlargest(8, "score_if")[
    ["id_contrato", "nombre_entidad", "modalidad_grupo", "valor_del_contrato",
     "rank_if", "rank_logvalor", "top1_if", "flag_p99_modalidad"]
])
display(exc)


## 4. Publicación

Se identifican los archivos generados con las puntuaciones y los contratos excluidos. La publicación en PostgreSQL permanece desactivada y solo podrá realizarse después de verificar los resultados y habilitar expresamente la escritura.

In [ ]:
print("No se publicó. Para publicar, ESCRIBIR=True tras revisar docs/11.")
print("CSV:", SALIDAS / "publicacion_scores.csv")
print("Excluidos:", SALIDAS / "publicacion_excluidos.csv")
